# Proxmox Datacenter Manager Analytics Dashboard

This notebook connects to the local PDM backend API, retrieves authenticated allocation metrics for the 10 mock server clusters, and visualizes CPU, Memory, and Storage allocations using Matplotlib and Pandas.

In [1]:
import urllib.request
import json
import pandas as pd
import matplotlib.pyplot as plt

# 1. Log in to get the access token
login_url = "http://localhost:8000/api/auth/login"
dashboard_url = "http://localhost:8000/api/proxmox/dashboard"

login_payload = json.dumps({"username": "admin", "password": "admin123"}).encode('utf-8')

try:
    # Authenticate
    req = urllib.request.Request(login_url, data=login_payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        token = json.loads(resp.read().decode())["token"]
        print("Successfully authenticated with PDM API.")
        
    # Fetch dashboard data
    req2 = urllib.request.Request(dashboard_url, headers={"Authorization": f"Bearer {token}"})
    with urllib.request.urlopen(req2) as resp2:
        dashboard_data = json.loads(resp2.read().decode())
        print(f"Fetched details for {len(dashboard_data['servers'])} clusters.")
except Exception as e:
    print(f"Error connecting to backend: {e}")

ModuleNotFoundError: No module named 'pandas'

## Parse and Process Cluster Allocation Metrics

We extract allocation percentages, nodes, and guest VM counts into a Pandas DataFrame for analysis.

In [ ]:
# Parse the cluster metrics
rows = []
for s in dashboard_data["servers"]:
    rows.append({
        "Cluster Name": s["name"],
        "Status": s["status"],
        "Nodes Online": f"{s['summary']['online_nodes']}/{s['summary']['total_nodes']}",
        "Active Guests": f"{s['summary']['running_guests']}/{s['summary']['total_guests']}",
        "Memory Alloc (%)": s["resources"]["memory_allocated_pct"],
        "vCPU Alloc (%)": s["cpu_virtualization"]["utilization_pct"] if s.get("cpu_virtualization") else 0,
        "Storage Alloc (%)": s["resources"]["storage_allocated_pct"],
        "Overcommit Limit (%)": s["cpu_overcommit_ratio"]
    })

df = pd.DataFrame(rows)
df

## Visualizing Allocation Comparisons

We plot the Memory, vCPU, and Storage allocation rates of each cluster side-by-side using Matplotlib.

In [ ]:
# Set plot style and figures
plt.figure(figsize=(12, 6))

x = range(len(df))
width = 0.25

plt.bar([i - width for i in x], df["Memory Alloc (%)"], width, label="Memory Allocation (%)", color="#8b5cf6")
plt.bar(x, df["vCPU Alloc (%)"], width, label="vCPU Allocation (%)", color="#3b82f6")
plt.bar([i + width for i in x], df["Storage Alloc (%)"], width, label="Storage Allocation (%)", color="#ec4899")

plt.xticks(x, df["Cluster Name"], rotation=45, ha="right")
plt.ylabel("Allocation Rate (%)")
plt.title("Proxmox Cluster Capacity Allocation Rates")
plt.legend()
plt.tight_layout()
plt.show()